# 🌾 Operation CropWatch — beta storyline

**Audience:** you, deciding whether this story is the right spine for the workshop.
**Runs on:** a laptop/CPU, no GPU, in a few minutes once the data is cached.

## The mission

The most widely used global cropland map, **GFSAD1000**, is a snapshot of **2010**. A food-security
agency wants to answer a simple question for the next lean season:

> *Which cropland is **irrigated** (buffered against a failed rainy season) and which is **rainfed** (exposed to it)?*

The 2010 map cannot tell us what has changed since. So instead of trusting a 15-year-old map, we will
**train a model that predicts irrigated-vs-rainfed from satellite and climate signals**, signals that
are refreshed every year, and then re-run it on new years.

This notebook walks the whole **data-science model lifecycle**:

| # | Stage | Question it answers |
|---|---|---|
| 1 | Get the data | What do we have, and where does it come from? |
| 2 | Clean and explore | Is the data usable, and does the physics make sense? |
| 3 | Baseline | What does a quick model say? *(spoiler: it flatters us)* |
| 4 | Honest evaluation | How good is it really? |
| 5 | Model and tune | Can a neural net do better, and with which settings? |
| 6 | Final training and test | What do we report? |
| 7 | Does it transfer? | Where must we **not** use it? |
| 8 | Package | How do we hand the model to someone else? |
| 9 | Gate and automate | How do we stop a bad model from shipping? |
| 10 | Diagnose, fix, refresh | The gate objected: why? Then re-run on 2024 data. The payoff. |
| 11 | Serve | How does an app call it? |
| 12 | Retrospective | What was painful? How long did each stage take? |

**Honest caveats up front:** the labels are themselves a *modelled* 1 km product (not field truth),
there is no ground-truth for 2024, and the task is binary (classes 1–2 → *irrigated*, 3–5 → *rainfed*;
non-cropland is dropped).

In [ ]:
import os, time, json, math, warnings
from pathlib import Path
from contextlib import contextmanager

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

import torch, torch.nn as nn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, recall_score,
                             precision_score, average_precision_score, confusion_matrix,
                             precision_recall_curve)

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(min(8, os.cpu_count() or 1))
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})

DATA_DIR = Path('data')
ART_DIR = Path('artifacts/cropwatch_v1')
EE_PROJECT = os.environ.get('EE_PROJECT', 'YOUR-GCP-PROJECT-ID')          # your Google Cloud project, only needed if the cache is empty

# bounding boxes: (west, south, east, north)
REGIONS = {'igp': (75, 26, 85, 31),      # Indo-Gangetic Plain: where we train
           'sahel': (3, 11, 9, 14)}      # West African Sahel: where we test transfer

timings = {}
@contextmanager
def stage(name):
    t0 = time.time(); yield
    timings[name] = time.time() - t0
    print(f'[{name}] {timings[name]:.1f}s')

print('torch', torch.__version__, '| CPU threads', torch.get_num_threads(), '| GPU available:', torch.cuda.is_available())

In [ ]:
os.listdir(DATA_DIR)

## 1 · Meet the data

**Labels, GFSAD1000 v1 (USGS/NASA):** one band, `landcover`, at ~1 km, nominal year 2010.

| value | meaning | our binary label |
|---|---|---|
| 0 | Non-cropland | *dropped* |
| 1 | Irrigation, major | **irrigated** |
| 2 | Irrigation, minor | **irrigated** |
| 3 | Rainfed | rainfed |
| 4 | Rainfed, minor fragments | rainfed *(noisy: mixed pixels)* |
| 5 | Rainfed, very minor fragments | rainfed *(noisy: mixed pixels)* |

GFSAD1000 has **only labels, no features**. We join signals from other public Earth Engine datasets
onto the same 1 km grid:

| Feature | Source | What it tells us |
|---|---|---|
| `ndvi_01 … 12` | MODIS MOD13A2 | monthly vegetation greenness |
| `rain_01 … 12` | CHIRPS | monthly rainfall (mm) |
| `temp_01 … 12` | ERA5-Land | monthly mean air temperature (°C) |
| `elevation` | SRTM | metres |

Irrigated fields should stay **green in the dry season**; rainfed fields should only green up when it rains.
That is the physical signal we are hoping a model can learn.

The next cell is the data-download code. It runs **once** and caches Parquet files under `data/`.
After that the notebook never touches the network.

In [ ]:
import math, time, os
import numpy as np
import pandas as pd
import ee

GRID_STEP = 0.008928197055985392      # GFSAD1000 native pixel size (degrees, ~1 km)
GRID_X0, GRID_Y0 = -180.254454098528, 90.25445409852799
MONTHS = range(1, 13)

def feature_image(year):
    """One image: 12x NDVI, 12x rainfall, 12x temperature, elevation, plus the GFSAD label."""
    start = ee.Date.fromYMD(year, 1, 1)
    ndvi = ee.ImageCollection('MODIS/061/MOD13A2').select('NDVI')
    rain = ee.ImageCollection('UCSB-CHG/CHIRPS/PENTAD')
    temp = ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR').select('temperature_2m')
    bands = []
    for m in MONTHS:
        a = start.advance(m - 1, 'month'); b = a.advance(1, 'month')
        bands.append(ndvi.filterDate(a, b).mean().multiply(0.0001).rename(f'ndvi_{m:02d}'))
        bands.append(rain.filterDate(a, b).sum().rename(f'rain_{m:02d}'))
        bands.append(temp.filterDate(a, b).mean().subtract(273.15).rename(f'temp_{m:02d}'))
    elev = (ee.Image('USGS/SRTMGL1_003')
            .reduceResolution(ee.Reducer.mean(), maxPixels=2048)
            .reproject(crs='EPSG:4326', scale=1000).rename('elevation'))
    label = ee.Image('USGS/GFSAD1000_V1').select('landcover').rename('landcover')
    return ee.Image.cat(bands + [elev, label]).toFloat()

def pull_tile(img, row0, col0, h, w, retries=3):
    grid = {'dimensions': {'width': w, 'height': h}, 'crsCode': 'EPSG:4326',
            'affineTransform': {'scaleX': GRID_STEP, 'shearX': 0, 'translateX': GRID_X0 + col0 * GRID_STEP,
                                'shearY': 0, 'scaleY': -GRID_STEP, 'translateY': GRID_Y0 - row0 * GRID_STEP}}
    for attempt in range(retries):
        try:
            return ee.data.computePixels({'expression': img, 'fileFormat': 'NUMPY_NDARRAY', 'grid': grid})
        except Exception as e:
            if attempt == retries - 1: raise
            time.sleep(5 * (attempt + 1))

def build_region(bbox, year, out_path, tile=256):
    """Download a bounding box on the GFSAD 1 km grid as a tidy table (one row per pixel) and cache it."""
    if os.path.exists(out_path):
        return pd.read_parquet(out_path)
    west, south, east, north = bbox
    col0 = math.floor((west - GRID_X0) / GRID_STEP); row0 = math.floor((GRID_Y0 - north) / GRID_STEP)
    ncol = math.ceil((east - west) / GRID_STEP);     nrow = math.ceil((north - south) / GRID_STEP)
    img = feature_image(year)
    frames = []
    for r in range(0, nrow, tile):
        for c in range(0, ncol, tile):
            h, w = min(tile, nrow - r), min(tile, ncol - c)
            arr = pull_tile(img, row0 + r, col0 + c, h, w)
            rr, cc = np.mgrid[r:r + h, c:c + w]
            df = pd.DataFrame({n: arr[n].ravel() for n in arr.dtype.names})
            df.insert(0, 'lat', GRID_Y0 - (row0 + rr.ravel() + 0.5) * GRID_STEP)
            df.insert(0, 'lon', GRID_X0 + (col0 + cc.ravel() + 0.5) * GRID_STEP)
            frames.append(df)
            print(f'  tile rows {r}-{r+h}, cols {c}-{c+w} done')
    out = pd.concat(frames, ignore_index=True)
    out = out[out['landcover'].notna()].reset_index(drop=True)
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    out.to_parquet(out_path)
    return out

In [ ]:
files = {'igp2010': ('igp', 2010),      # training region, features from the label year
         'sahel2010': ('sahel', 2010),  # held-out region, features from the label year
         'igp2024': ('igp', 2024)}      # same region, new year: no labels exist

if not all((DATA_DIR / f'{k}.parquet').exists() for k in files):
    ee.Initialize(project=EE_PROJECT)    # first run only (about 5 minutes for all three)

with stage('1 get data'):
    raw = {k: build_region(REGIONS[r], y, str(DATA_DIR / f'{k}.parquet')) for k, (r, y) in files.items()}

for k, d in raw.items():
    print(f'{k:10s} {d.shape[0]:>8,} pixels x {d.shape[1]} columns')
raw['igp2010'].iloc[:3, [0, 1, 2, 3, 4, -2, -1]]

## 2 · Clean and explore

Two jobs: make the table model-ready, and check that the data tells a believable story before we model anything.

In [ ]:
BASE = [f'{p}_{m:02d}' for m in range(1, 13) for p in ('ndvi', 'rain', 'temp')] + ['elevation']
DRY = [f'ndvi_{m:02d}' for m in range(1, 6)]            # Jan-May, little rain in both regions

def add_features(df):
    # Hand-made summaries. At serving time this exact function must run again (see section 11).
    out = df.copy()
    nd = df[[f'ndvi_{m:02d}' for m in range(1, 13)]]
    out['ndvi_max'] = nd.max(axis=1)
    out['ndvi_amp'] = nd.max(axis=1) - nd.min(axis=1)
    out['ndvi_dry'] = df[DRY].mean(axis=1)                       # greenness while it is dry
    if all(f'rain_{m:02d}' in df for m in range(1, 13)):         # rainfall summaries only if rainfall was supplied
        rn = df[[f'rain_{m:02d}' for m in range(1, 13)]]
        out['rain_year'] = rn.sum(axis=1)
        out['rain_monsoon'] = rn[[f'rain_{m:02d}' for m in range(6, 10)]].sum(axis=1)
        out['rain_dry'] = out['rain_year'] - out['rain_monsoon']
    return out

ENG = ['ndvi_max', 'ndvi_amp', 'ndvi_dry', 'rain_year', 'rain_monsoon', 'rain_dry']
FEATURES = BASE + ENG

def clean(df):
    df = df[df['landcover'] > 0]                                  # cropland only
    ok = np.isfinite(df[BASE].astype('float64')).all(axis=1)      # NDVI sometimes comes back as -inf
    out = add_features(df[ok]).reset_index(drop=True)
    out['y'] = (out['landcover'] <= 2).astype('float32')          # 1 = irrigated, 0 = rainfed
    return out, int((~ok).sum())

data = {}
for k, d in raw.items():
    data[k], n_bad = clean(d)
    print(f'{k:10s} cropland pixels: {len(data[k]):>8,} | dropped for bad values: {n_bad:>3} | irrigated share: {data[k].y.mean():.1%}')

print()
print(pd.crosstab(raw['igp2010'].landcover.map({0:'0 non-crop',1:'1 irrig major',2:'2 irrig minor',3:'3 rainfed',4:'4 rainfed frag',5:'5 rainfed v.frag'}).rename('IGP'), 'pixels').T.to_string())
print(pd.crosstab(raw['sahel2010'].landcover.map({0:'0 non-crop',1:'1 irrig major',2:'2 irrig minor',3:'3 rainfed',4:'4 rainfed frag',5:'5 rainfed v.frag'}).rename('Sahel'), 'pixels').T.to_string())

**Notice:** the two regions are very different worlds. In the Indo-Gangetic Plain most cropland is irrigated
(about 7 in 10 pixels). In the Sahel it is the opposite, with irrigation under 10%. We deliberately train on the first
and keep the second as a stress test.

In [ ]:
def to_grid(df, col, bbox):
    # Rebuild a raster from the table so we can draw maps (pixels are on the GFSAD grid).
    c = np.floor((df['lon'].values - GRID_X0) / GRID_STEP).astype(int)
    r = np.floor((GRID_Y0 - df['lat'].values) / GRID_STEP).astype(int)
    g = np.full((r.max() - r.min() + 1, c.max() - c.min() + 1), np.nan)
    g[r - r.min(), c - c.min()] = df[col].values
    return g

def label_view(df):                                           # 0 non-crop, 1 rainfed, 2 irrigated
    return df.assign(v=np.select([df.landcover == 0, df.landcover >= 3], [0, 1], 2))

cmap = ListedColormap(['#e8e4d8', '#e0a030', '#2a6fb0'])
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2), gridspec_kw={'width_ratios': [2, 1.1]})
for a, (name, k) in zip(ax, [('Indo-Gangetic Plain (train)', 'igp2010'), ('West African Sahel (held out)', 'sahel2010')]):
    a.imshow(to_grid(label_view(raw[k]), 'v', None), cmap=cmap, vmin=0, vmax=2, interpolation='nearest'); a.set_title(name); a.axis('off')
from matplotlib.patches import Patch
fig.legend(handles=[Patch(color='#e8e4d8', label='non-cropland'), Patch(color='#e0a030', label='rainfed'), Patch(color='#2a6fb0', label='irrigated')],
           loc='lower center', ncol=3, frameon=False)
plt.tight_layout(rect=(0, 0.06, 1, 1)); plt.show()

In [ ]:
months = np.arange(1, 13)
fig, ax = plt.subplots(2, 2, figsize=(11, 6), sharex=True)
for j, (name, k) in enumerate([('Indo-Gangetic Plain', 'igp2010'), ('Sahel', 'sahel2010')]):
    d = data[k]
    for lab, colr, nm in [(1, '#2a6fb0', 'irrigated'), (0, '#e0a030', 'rainfed')]:
        sub = d[d.y == lab]
        ax[0, j].plot(months, [sub[f'ndvi_{m:02d}'].mean() for m in months], color=colr, lw=2.2, label=f'{nm} (n={len(sub):,})')
        ax[1, j].plot(months, [sub[f'rain_{m:02d}'].mean() for m in months], color=colr, lw=2.2)
    ax[0, j].set_title(name); ax[0, j].legend(frameon=False, fontsize=8)
ax[0, 0].set_ylabel('NDVI (greenness)'); ax[1, 0].set_ylabel('rainfall (mm / month)')
for a in ax[1]: a.set_xlabel('month'); a.set_xticks(months)
plt.tight_layout(); plt.show()

**Does the physics make sense?** In the Indo-Gangetic Plain the irrigated curve peaks in **January–February** (winter wheat watered by canals and wells)
when there is almost no rain, while rainfed land is flat then and only greens with the summer monsoon. That winter gap is the signal. In the Sahel the rainfall regime is entirely different (one short wet season, almost no rain the rest of the year), so
the curves live in another range. Keep that in mind for section 7.

## 3 · The tempting baseline

Quick and standard: take all pixels, shuffle them, put 80% in training and 20% in test, and fit a random forest.

In [ ]:
def metrics(y, prob, thr=0.5):
    p = (prob >= thr).astype(int)
    return {'accuracy': accuracy_score(y, p), 'balanced_acc': balanced_accuracy_score(y, p),
            'irrigated_recall': recall_score(y, p), 'irrigated_precision': precision_score(y, p, zero_division=0),
            'avg_precision': average_precision_score(y, prob)}

igp = data['igp2010']
rng = np.random.RandomState(SEED)
perm = rng.permutation(len(igp)); cut = int(0.8 * len(igp))
shuf_tr, shuf_te = igp.iloc[perm[:cut]], igp.iloc[perm[cut:]]

with stage('3 baseline random forest'):
    rf_shuffled = RandomForestClassifier(60, n_jobs=-1, random_state=SEED).fit(shuf_tr[BASE], shuf_tr.y)
base_scores = {'always say "irrigated"': metrics(shuf_te.y, np.ones(len(shuf_te))),
               'random forest, shuffled 80/20 split': metrics(shuf_te.y, rf_shuffled.predict_proba(shuf_te[BASE])[:, 1])}
pd.DataFrame(base_scores).T.round(3)

**91% accuracy**, balanced accuracy near 0.89. Ship it? Two red flags before anyone celebrates:

1. **"Always say irrigated" already scores 72%.** Accuracy is a poor yardstick when classes are lopsided. We should watch
   *balanced* accuracy and *irrigated recall*.
2. **Neighbouring pixels are near-duplicates** (same weather, same canal, same crop calendar). Shuffling puts a pixel's twin
   in training and tests on its sibling, so the model is partly *remembering the neighbourhood*, not learning irrigation.

## 4 · Honest evaluation: split by place, not by pixel

We cut the map into 1° × 1° blocks (~100 km) and assign **whole blocks** to train (60%), validation (20%) and test (20%).
Validation is for tuning; test is touched once at the end. A **second, much harder test** is the Sahel: a different continent
the model never sees.

In [ ]:
igp['block'] = np.floor(igp.lon).astype(int) * 1000 + np.floor(igp.lat).astype(int)
blocks = np.array(sorted(igp.block.unique())); rs = np.random.RandomState(7); rs.shuffle(blocks)
n = len(blocks); cut1, cut2 = int(.6 * n), int(.8 * n)
role = {b: 'train' for b in blocks[:cut1]} | {b: 'val' for b in blocks[cut1:cut2]} | {b: 'test' for b in blocks[cut2:]}
igp['split'] = igp.block.map(role)
tr, va, te = (igp[igp.split == s].reset_index(drop=True) for s in ('train', 'val', 'test'))
sa = data['sahel2010']
print(pd.DataFrame({'blocks': pd.Series(role).value_counts(), 'pixels': igp.split.value_counts(),
                    'irrigated share': igp.groupby('split').y.mean().round(3)}).loc[['train', 'val', 'test']].to_string())

fig, ax = plt.subplots(figsize=(7, 3.4))
g = to_grid(igp.assign(s=igp.split.map({'train': 0, 'val': 1, 'test': 2})), 's', None)
ax.imshow(g, cmap=ListedColormap(['#9bb7d4', '#e5b84d', '#d9625a']), interpolation='nearest'); ax.axis('off')
ax.set_title('Spatial split of the Indo-Gangetic Plain  (blue = train, yellow = validation, red = test)', fontsize=9); plt.show()

In [ ]:
with stage('4 honest evaluation (RF on spatial split)'):
    rf_spatial = RandomForestClassifier(60, n_jobs=-1, random_state=SEED).fit(tr[BASE], tr.y)
rows = dict(base_scores)
rows['RF, shuffled split (what we believed)'] = rows.pop('random forest, shuffled 80/20 split')
rows['RF, spatial validation blocks'] = metrics(va.y, rf_spatial.predict_proba(va[BASE])[:, 1])
rows['RF on the Sahel (never seen)'] = metrics(sa.y, rf_spatial.predict_proba(sa[BASE])[:, 1])
pd.DataFrame(rows).T.drop(index='always say "irrigated"').round(3)

**This is the finding of the notebook.** Same model, three very different answers:

* Shuffled split: balanced accuracy ≈ 0.89 *(flattering)*
* New blocks of the same region: ≈ 0.80 *(the honest in-region number)*
* The Sahel: **≈ 0.48, no better than a coin flip**, with irrigated recall near 0.14 and precision under 0.1. The model is effectively blind to irrigation there.

A better algorithm will not fix a bad evaluation. So from now on every number comes from **held-out places**.

## 5 · A neural network, and tuning it

Now the model we will carry forward: a small **MLP** (multi-layer perceptron) in PyTorch on the 37 raw features plus
6 hand-made ones (peak greenness, seasonal amplitude, dry-season greenness, annual / monsoon / dry rainfall). Inputs are standardised using
**training-set** statistics only. The settings (width, depth, dropout, learning rate, class weight) are what we tune.

We bundle "a feature set + its scaling + its tensors" into a **variant**, so that later we can compare different feature sets cleanly.

In [ ]:
def make_variant(feats):
    mu, sd = tr[feats].mean(), tr[feats].std() + 1e-6
    f = lambda d: torch.tensor(((d[feats] - mu) / sd).values, dtype=torch.float32)
    return {'feats': feats, 'mu': mu, 'sd': sd, 'f': f,
            'lo': tr[feats].quantile(0.01), 'hi': tr[feats].quantile(0.99),     # range seen in training (for the "unfamiliar pixel" check)
            'ood_min': max(3, round(0.28 * len(feats))),                        # unfamiliar = >= 28% of features outside that range
            'X_tr': f(tr), 'X_va': f(va), 'X_te': f(te), 'X_sa': f(sa)}

V1 = make_variant(FEATURES)
y_tr = torch.tensor(tr.y.values)

class CropNet(nn.Module):
    def __init__(self, n_in, width=64, depth=2, dropout=0.1):
        super().__init__()
        layers, i = [], n_in
        for _ in range(depth):
            layers += [nn.Linear(i, width), nn.ReLU(), nn.Dropout(dropout)]; i = width
        self.net = nn.Sequential(*layers, nn.Linear(i, 1))
    def forward(self, x): return self.net(x).squeeze(1)

def predict_proba(model, X):
    model.eval()
    with torch.no_grad(): return torch.sigmoid(model(X)).numpy()

def train_model(V, width=64, depth=2, dropout=0.1, lr=1e-3, pos_weight=1.0, epochs=6, batch=1024, seed=SEED, track=False):
    torch.manual_seed(seed)
    model = CropNet(V['X_tr'].shape[1], width, depth, dropout)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(float(pos_weight)))
    history, best = [], (-1, None)
    for ep in range(epochs):
        model.train(); order = torch.randperm(len(V['X_tr'])); total = 0.0
        for i in range(0, len(order), batch):
            j = order[i:i + batch]; opt.zero_grad()
            loss = loss_fn(model(V['X_tr'][j]), y_tr[j]); loss.backward(); opt.step(); total += loss.item() * len(j)
        if track:
            vb = metrics(va.y, predict_proba(model, V['X_va']))['balanced_acc']
            history.append({'epoch': ep + 1, 'train_loss': total / len(order), 'val_balanced_acc': vb})
            if vb > best[0]: best = (vb, {k: v.clone() for k, v in model.state_dict().items()})
    if track and best[1] is not None: model.load_state_dict(best[1])      # keep the best epoch
    return (model, pd.DataFrame(history)) if track else model

with stage('5 one MLP, default settings'):
    m0 = train_model(V1)
print('default MLP on validation blocks:', {k: round(v, 3) for k, v in metrics(va.y, predict_proba(m0, V1['X_va'])).items()})

### Tuning by random search

Each **trial** is one full training run with a different combination of settings, scored on the **validation blocks**
(balanced accuracy). Trials are completely independent of each other, a detail worth remembering when we
discuss runtime in section 12.

In [ ]:
N_TRIALS = 10
space_rng = np.random.RandomState(SEED)
def sample_config():
    return dict(width=int(space_rng.choice([32, 64, 128, 256])), depth=int(space_rng.choice([1, 2, 3])),
                dropout=float(space_rng.choice([0.0, 0.1, 0.3])), lr=float(10 ** space_rng.uniform(-3, -2)),
                pos_weight=float(space_rng.choice([0.5, 0.75, 1.0, 1.5])))

trials = []
with stage('5 random search (10 trials, run one after another)'):
    for t in range(N_TRIALS):
        cfg = sample_config(); t0 = time.time()
        m = train_model(V1, **cfg, seed=SEED + t)
        sc = metrics(va.y, predict_proba(m, V1['X_va']))
        trials.append({'trial': t, **cfg, 'val_balanced_acc': sc['balanced_acc'], 'val_avg_precision': sc['avg_precision'],
                       'seconds': time.time() - t0})
        print(f"trial {t}: width={cfg['width']:>3} depth={cfg['depth']} drop={cfg['dropout']:.1f} lr={cfg['lr']:.4f} pos_w={cfg['pos_weight']:.2f}"
              f"  ->  val balanced acc {sc['balanced_acc']:.3f}  ({trials[-1]['seconds']:.0f}s)")

results = pd.DataFrame(trials).sort_values('val_balanced_acc', ascending=False).reset_index(drop=True)
best_cfg = {k: results.loc[0, k].item() for k in ['width', 'depth', 'dropout', 'lr', 'pos_weight']}   # plain Python numbers
best_cfg['width'], best_cfg['depth'] = int(best_cfg['width']), int(best_cfg['depth'])
print('\nbest settings:', best_cfg)
results.round(3)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].scatter(results.seconds, results.val_balanced_acc, s=60, c=results.pos_weight, cmap='viridis')
ax[0].set_xlabel('training time per trial (s)'); ax[0].set_ylabel('validation balanced accuracy'); ax[0].set_title('cost vs quality (colour = class weight)')
ax[1].scatter(results.pos_weight, results.val_balanced_acc, s=60, color='#2a6fb0')
ax[1].set_xlabel('class weight on "irrigated"'); ax[1].set_title('which knob mattered?'); plt.tight_layout(); plt.show()

In this run, **down-weighting the majority class** (`pos_weight` below 1; irrigated is the majority in training) gave the best balanced accuracy,
and the biggest, slowest networks were not the winners. Two lessons: tuning is **cheap exploration, not magic**, and the most expensive trial is not
necessarily the best one.

## 6 · Final training and the test we only run once

Retrain with the winning settings for longer (15 epochs), keep the best epoch by validation score, then run the **untouched test blocks**.

In [ ]:
with stage('6 final training (15 epochs)'):
    final, history = train_model(V1, **best_cfg, epochs=15, track=True)

fig, ax = plt.subplots(figsize=(6.5, 3.2)); ax.plot(history.epoch, history.train_loss, color='#555', label='train loss')
ax2 = ax.twinx(); ax2.plot(history.epoch, history.val_balanced_acc, color='#2a6fb0', label='validation balanced acc')
ax.set_xlabel('epoch'); ax.set_ylabel('train loss'); ax2.set_ylabel('val balanced acc'); ax.set_title('Learning curve'); plt.show()

p_te, p_sa = predict_proba(final, V1['X_te']), predict_proba(final, V1['X_sa'])
report = {'validation blocks': metrics(va.y, predict_proba(final, V1['X_va'])),
          'TEST blocks (same region)': metrics(te.y, p_te),
          'Sahel (new region)': metrics(sa.y, p_sa)}
compare = {**{f'MLP: {k}': v for k, v in report.items()},
           'random forest (section 4): TEST blocks': metrics(te.y, rf_spatial.predict_proba(te[BASE])[:, 1]),
           'always say "irrigated": TEST blocks': metrics(te.y, np.ones(len(te)))}
pd.DataFrame(compare).T.round(3)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
for a, (nm, y, p) in zip(ax[:2], [('Test blocks', te.y, p_te), ('Sahel', sa.y, p_sa)]):
    cm = confusion_matrix(y, p >= 0.5); a.imshow(cm, cmap='Blues'); a.set_title(nm)
    a.set_xticks([0, 1], ['pred rainfed', 'pred irrigated']); a.set_yticks([0, 1], ['rainfed', 'irrigated'])
    for (i, j), v in np.ndenumerate(cm): a.text(j, i, f'{v:,}', ha='center', va='center', color='white' if v > cm.max() / 2 else 'black')
for nm, y, p in [('Test blocks', te.y, p_te), ('Sahel', sa.y, p_sa)]:
    pr, rc, _ = precision_recall_curve(y, p); ax[2].plot(rc, pr, label=f'{nm} (AP {average_precision_score(y, p):.2f})')
    ax[2].axhline(y.mean(), ls=':', lw=1, color='grey')
ax[2].set_xlabel('recall of irrigated'); ax[2].set_ylabel('precision'); ax[2].set_title('Precision-recall'); ax[2].legend(frameon=False)
plt.tight_layout(); plt.show()

Read this table carefully, because it is not flattering to the neural net:

* **Better than guessing, yes:** on unseen blocks of the home region, balanced accuracy is ≈ 0.74 against 0.50 for "always irrigated".
* **Not better than the random forest.** The forest from section 4 scores about the same on balanced accuracy (≈ 0.75) and clearly higher on average precision.
  On a small tabular problem like this one that is common; a team judging on accuracy alone would simply keep the forest.
* **The validation score (0.81) is optimistic.** We chose the settings and the best epoch *on* validation. The test number (≈ 0.74) is the one to quote. The gap
  also reflects that blocks differ: only ~60% of validation pixels are irrigated against ~72% in test.

We carry the MLP forward for the rest of the story because it is the model that uses more data and compute and gets tuned; the forest stays as the benchmark to beat.
We will call this model **v1**.

## 7 · Does it transfer?

The Sahel result above shows it does not. Instead of just saying so, let's **measure why**, so a machine can warn us in future.
For each pixel, count how many features fall outside the range the model saw in training (its 1st to 99th percentile).

In [ ]:
def n_outside(df, V):
    f = V['feats']
    return ((df[f] < V['lo']) | (df[f] > V['hi'])).sum(axis=1)

shift = {'IGP test blocks': n_outside(te, V1), 'IGP 2024 (same place, new year)': n_outside(data['igp2024'], V1), 'Sahel 2010': n_outside(sa, V1)}
print(f"a pixel is 'unfamiliar' if >= {V1['ood_min']} of its {len(V1['feats'])} features lie outside the training range")
pd.DataFrame({'median features outside range': {k: v.median() for k, v in shift.items()},
              'share of unfamiliar pixels': {k: (v >= V1['ood_min']).mean() for k, v in shift.items()}}).round(3)

In the Sahel nearly every pixel is unfamiliar: different rainfall regime, different temperatures, a different crop calendar.
This is **covariate shift**, and the model will confidently give wrong answers there. The same place in a *new year* (IGP 2024) looks familiar.
This gives us a rule we can automate: *"don't trust predictions on pixels that look nothing like the training data."*

## 8 · Package the model

A model is not a `.pt` file. To hand it to someone else we also need the **feature order, scaling statistics, thresholds,
the training-data ranges for the "unfamiliar pixel" check, and the metrics we promised.** We save all of it together.

In [ ]:
def save_artifact(V, model, cfg, report, version, n_train):
    path = Path('artifacts') / f'cropwatch_{version}'; path.mkdir(parents=True, exist_ok=True)
    feats = V['feats']
    torch.save({'state_dict': model.state_dict(),
                'arch': {'n_in': len(feats), 'width': cfg['width'], 'depth': cfg['depth'], 'dropout': cfg['dropout']}}, path / 'model.pt')
    meta = {'name': 'cropwatch_irrigated_vs_rainfed', 'version': version, 'created': pd.Timestamp.now().isoformat(timespec='seconds'),
            'task': 'binary: 1 = irrigated (GFSAD classes 1-2), 0 = rainfed (classes 3-5); cropland pixels only',
            'required_inputs': [c for c in BASE if c in feats],             # what the caller must send
            'features': feats, 'mean': V['mu'].tolist(), 'std': V['sd'].tolist(),
            'train_p01': V['lo'].tolist(), 'train_p99': V['hi'].tolist(), 'ood_min_features': V['ood_min'],
            'threshold': 0.5, 'hyperparameters': cfg,
            'trained_on': {'region': 'Indo-Gangetic Plain', 'bbox': REGIONS['igp'], 'year': 2010, 'pixels': int(n_train)},
            'metrics': {k: {m: round(float(v), 4) for m, v in d.items()} for k, d in report.items()},
            'known_limits': ['labels are a modelled 1 km product, not field truth',
                             'does not transfer to regions with a different climate regime (see Sahel metrics)']}
    (path / 'metadata.json').write_text(json.dumps(meta, indent=1)); return path

with stage('8 package v1'):
    path_v1 = save_artifact(V1, final, best_cfg, report, 'v1', len(tr))
print(path_v1, {f.name: f'{f.stat().st_size / 1024:.0f} KB' for f in path_v1.iterdir()})

## 9 · Gate and automate

Everything so far was typed by hand. For the **annual refresh** we want a repeatable routine with a *gate* that refuses to publish a
map when something looks wrong. Three checks:

1. **Quality:** the model's held-out test scores clear a minimum.
2. **Familiarity:** the new inputs look like the data the model was trained on (section 7).
3. **Stability:** the model's overall answer does not swing wildly between the reference year and the new year. This is *prediction drift* monitoring.

The thresholds are a **policy choice** a team would agree *before* shipping; here we set them a little below v1's measured test scores.

In [ ]:
GATE = {'min_test_balanced_acc': 0.70, 'min_test_avg_precision': 0.80, 'max_unfamiliar_share': 0.10, 'max_prediction_shift': 0.10}

def irrigated_share(model, V, df): return float((predict_proba(model, V['f'](df)) >= 0.5).mean())

def deployment_gate(V, model, report, new_df, reference_df):
    unfamiliar = float((n_outside(new_df, V) >= V['ood_min']).mean())
    ref, new = irrigated_share(model, V, reference_df), irrigated_share(model, V, new_df)
    t = report['TEST blocks (same region)']; reasons = []
    if t['balanced_acc'] < GATE['min_test_balanced_acc']: reasons.append('test balanced accuracy too low')
    if t['avg_precision'] < GATE['min_test_avg_precision']: reasons.append('test average precision too low')
    if unfamiliar > GATE['max_unfamiliar_share']: reasons.append(f'{unfamiliar:.0%} of new pixels look unlike the training data')
    if abs(new - ref) > GATE['max_prediction_shift']: reasons.append(f'predicted irrigated share moved {ref:.0%} -> {new:.0%}')
    return {'passed': not reasons, 'unfamiliar_share': round(unfamiliar, 3), 'share_reference': round(ref, 3), 'share_new': round(new, 3), 'reasons': reasons}

def refresh_map(V, model, report, new_df, reference_df, label):
    gate = deployment_gate(V, model, report, new_df, reference_df)
    print(f'{label}: gate ->', 'PASS' if gate['passed'] else 'BLOCKED'); print('   ', gate)
    if not gate['passed']: return None
    out = new_df[['lon', 'lat']].copy(); out['p_irrigated'] = predict_proba(model, V['f'](new_df)); return out

print('thresholds:', GATE, '\n')
with stage('9 gate checks (v1)'):
    _ = refresh_map(V1, final, report, data['igp2024'], igp, 'v1 on Indo-Gangetic Plain 2024')
    _ = refresh_map(V1, final, report, sa, sa, 'v1 on the Sahel')

Two different alarms, both useful:

* **Sahel:** blocked because the inputs are unfamiliar, exactly as section 7 predicted. That region needs its own model and local labels.
* **Indo-Gangetic Plain 2024:** the inputs look familiar and the test scores are fine, yet the gate is *still* blocked. The model's overall answer swings
  by far more than 14 years of irrigation change plausibly could. **Something is making v1 fragile.** A quality-only gate would have shipped this map.

## 10 · Diagnose the alarm, fix the model, then refresh

**Hypothesis:** a single year's temperature and rainfall describe *that year's weather*, not whether a field is irrigated. The 2010 model leaned on
them as shortcuts, and 2024 weather is different. Let's look at the evidence and then test it. Inputs for the same pixels:

In [ ]:
key = lambda d: d.assign(k=d.lon.round(5).astype(str) + '_' + d.lat.round(5).astype(str)).set_index('k')
a_, b_ = key(igp), key(data['igp2024'])
common = a_.index.intersection(b_.index)                       # a few pixels are dropped in one year only, so align by location
pair10, pair24 = a_.loc[common], b_.loc[common]
print(f'pixels valid in both years: {len(common):,}\n')
months_cmp = pd.DataFrame({'2010': pair10[['temp_01', 'temp_03', 'temp_04', 'rain_03', 'rain_06', 'ndvi_02']].mean(),
                           '2024': pair24[['temp_01', 'temp_03', 'temp_04', 'rain_03', 'rain_06', 'ndvi_02']].mean()})
months_cmp['change'] = months_cmp['2024'] - months_cmp['2010']
months_cmp.round(2)

Same pixels, but spring 2024 was **several degrees cooler** and had a different rainfall pattern than spring 2010. If the model uses temperature and rain as
shortcuts, those shifts push its answers around. **Test:** retrain the same network on three feature sets, and for each look at test accuracy *and*
how much its answer swings between 2010 and 2024 inputs.

In [ ]:
def evaluate_variant(name, feats, reuse=None):
    V = reuse[0] if reuse else make_variant(feats)
    model = reuse[1] if reuse else train_model(V, **best_cfg, epochs=15, track=True)[0]
    t = metrics(te.y, predict_proba(model, V['X_te']))
    c10, c24 = predict_proba(model, V['f'](pair10)) >= .5, predict_proba(model, V['f'](pair24)) >= .5
    v = metrics(va.y, predict_proba(model, V['X_va']))
    row = {'features': len(feats), 'val_balanced_acc': v['balanced_acc'], 'test_balanced_acc': t['balanced_acc'], 'test_avg_precision': t['avg_precision'],
           'irrigated share 2010 inputs': c10.mean(), 'irrigated share 2024 inputs': c24.mean(),
           'pixels that flip class': (c10 != c24).mean()}
    return name, row, V, model

no_temp = [c for c in FEATURES if not c.startswith('temp')]
ndvi_only = [c for c in FEATURES if c.startswith('ndvi') or c == 'elevation']
with stage('10 ablation (retrain 2 more variants)'):
    variants = [evaluate_variant('all features (v1)', FEATURES, reuse=(V1, final)),
                evaluate_variant('drop temperature', no_temp),
                evaluate_variant('greenness + elevation only', ndvi_only)]
ablation = pd.DataFrame({n: r for n, r, _, _ in variants}).T
ablation.round(3)

The pattern is clear: **removing temperature and rainfall costs nothing in accuracy** (it even helps slightly), but it makes the answer **far more stable across years**.
The classes are separated by *how green the land is through the dry season*, which is what irrigation actually does, and not by that year's weather.

We pick the variant that stays within a small accuracy tolerance of v1 **on validation** (the test blocks stay out of the decision) and swings least between years, and call it **v2**.

In [ ]:
v1_val = variants[0][1]['val_balanced_acc']
ok = [(abs(r['irrigated share 2024 inputs'] - r['irrigated share 2010 inputs']), n, V, m) for n, r, V, m in variants if r['val_balanced_acc'] >= v1_val - 0.02]
_, v2_name, V2, model_v2 = min(ok, key=lambda x: x[0])
print('v2 =', v2_name, '| features:', len(V2['feats']), '| needs inputs:', [c for c in BASE if c in V2['feats']][:3], '...')

report2 = {'validation blocks': metrics(va.y, predict_proba(model_v2, V2['X_va'])),
           'TEST blocks (same region)': metrics(te.y, predict_proba(model_v2, V2['X_te'])),
           'Sahel (new region)': metrics(sa.y, predict_proba(model_v2, V2['X_sa']))}
path_v2 = save_artifact(V2, model_v2, best_cfg, report2, 'v2', len(tr))
print('saved', path_v2, '\n')
map24 = refresh_map(V2, model_v2, report2, data['igp2024'], igp, 'v2 on Indo-Gangetic Plain 2024')
_ = refresh_map(V2, model_v2, report2, sa, sa, 'v2 on the Sahel')

v2 passes the gate for the same region and the new year, and **still refuses the Sahel** because far too many of its pixels look unfamiliar.

### The payoff: a 2024 map

No 2024 labels exist, so we cannot compute accuracy. We *can* compare the model with itself: where does its answer **change** between 2010 and 2024 inputs?
Model-vs-model comparison cancels the model's own systematic errors, so those flips are cleaner leads than disagreement with the old map.

In [ ]:
p10 = predict_proba(model_v2, V2['f'](pair10)); p24 = predict_proba(model_v2, V2['f'](pair24))
cmp = pd.DataFrame({'lon': pair24.lon.values, 'lat': pair24.lat.values, 'p10': p10, 'p24': p24, 'label10': pair10.y.values})
cmp['flip'] = np.select([(p10 >= .5) & (p24 < .5) & ((p10 - p24) > .4), (p10 < .5) & (p24 >= .5) & ((p24 - p10) > .4)], [-1, 1], 0)

print(pd.Series({'irrigated share, 2010 label': cmp.label10.mean(), 'irrigated share, v2 on 2010 inputs': (p10 >= .5).mean(),
                 'irrigated share, v2 on 2024 inputs': (p24 >= .5).mean(), 'pixels flipped with confidence': (cmp.flip != 0).mean(),
                 '  irrigated -> rainfed': (cmp.flip == -1).mean(), '  rainfed -> irrigated': (cmp.flip == 1).mean()}).round(4).to_string())

fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
ax[0].imshow(to_grid(cmp.assign(v=cmp.label10), 'v', None), cmap=ListedColormap(['#e0a030', '#2a6fb0']), interpolation='nearest'); ax[0].set_title('2010 label (GFSAD1000)')
im = ax[1].imshow(to_grid(cmp, 'p24', None), cmap='Blues', vmin=0, vmax=1, interpolation='nearest'); ax[1].set_title('v2: 2024 predicted P(irrigated)'); plt.colorbar(im, ax=ax[1], shrink=.8)
ax[2].imshow(to_grid(cmp.assign(v=cmp.flip + 1), 'v', None), cmap=ListedColormap(['#d9625a', '#e8e4d8', '#2a9d5c']), vmin=0, vmax=2, interpolation='nearest')
ax[2].set_title('confident flips 2010 -> 2024  (red: lost irrigation, green: gained)')
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()

**How to read this:** the overall irrigated share barely moves between the two years, and only a few percent of pixels flip with confidence, in *both* directions.
That is what you would expect from a canal-irrigated plain that has not been transformed, plus some noise. Check the right-hand map: flips that form **spatial clusters**
are more credible leads than isolated specks, which are more likely noise. In this run both colours form visible clusters rather than uniform specks, which is mildly
encouraging, but a cluster could equally be a regional weather effect (say, a late monsoon in one district).

Either way, flipped pixels are **leads for analysts or field teams**, not conclusions. There is no ground truth for 2024, a *single* year is still
noisy (a drought or a good monsoon year moves greenness), and the labels themselves are weak. A sensible next step is to use **multi-year averages**
rather than single years before anyone acts on a flip.

## 11 · Serve it

An app should not need our notebook. It sends a **JSON payload of one pixel's signals** and gets back a class and probability. Below, a tiny
`CropWatchService` loads *only the saved v2 artifact* (no variables from this notebook), re-applies **the same feature code**, and
refuses nonsense input. Running exactly the same preprocessing at serving as at training time matters: mismatch between the two is a classic silent failure.
Note the smaller payload: v2 only needs greenness and elevation, a nice side effect of dropping the weather features.

In [ ]:
class CropWatchService:
    def __init__(self, path):
        path = Path(path)
        self.meta = json.loads((path / 'metadata.json').read_text())
        ckpt = torch.load(path / 'model.pt', weights_only=True)
        self.model = CropNet(**ckpt['arch']); self.model.load_state_dict(ckpt['state_dict']); self.model.eval()
        self.feats = self.meta['features']
        self.mu, self.sd = np.array(self.meta['mean']), np.array(self.meta['std'])
        self.lo, self.hi = np.array(self.meta['train_p01']), np.array(self.meta['train_p99'])

    def predict(self, payload):
        df = pd.DataFrame(payload['instances'])
        missing = [c for c in self.meta['required_inputs'] if c not in df.columns]
        if missing: return {'error': f'missing inputs: {missing[:3]}{"..." if len(missing) > 3 else ""}'}
        nd = df[[f'ndvi_{m:02d}' for m in range(1, 13)]]
        if ((nd < -0.3) | (nd > 1.0)).any().any(): return {'error': 'NDVI outside the physical range [-0.3, 1.0]'}
        df = add_features(df)[self.feats]                      # the same feature code as training
        x = torch.tensor(((df.values - self.mu) / self.sd), dtype=torch.float32)
        with torch.no_grad(): p = torch.sigmoid(self.model(x)).numpy()
        unfamiliar = ((df.values < self.lo) | (df.values > self.hi)).sum(axis=1) >= self.meta['ood_min_features']
        return {'model': f"{self.meta['name']}:{self.meta['version']}",
                'predictions': [{'label': 'irrigated' if q >= self.meta['threshold'] else 'rainfed', 'p_irrigated': round(float(q), 3),
                                 'warning': 'pixel looks unlike the training data: do not trust' if u else None} for q, u in zip(p, unfamiliar)]}

service = CropWatchService(path_v2)                            # fresh object built from files only
needs = service.meta['required_inputs']
pick = lambda d: d[needs].iloc[0].round(4).to_dict()
payload = {'instances': [pick(te[te.y == 1]), pick(te[te.y == 0]), pick(sa[sa.y == 0])]}
print(f'request: {len(payload["instances"])} pixels x {len(needs)} inputs each, first input values:', {k: payload['instances'][0][k] for k in needs[:3]}, '...')
print('truth: irrigated (IGP), rainfed (IGP), rainfed (Sahel)\n')
print(json.dumps(service.predict(payload), indent=1))

In [ ]:
print('physically impossible NDVI ->', service.predict({'instances': [{**pick(te), 'ndvi_07': 7.5}]}))
print('missing inputs            ->', service.predict({'instances': [{'ndvi_01': 0.4}]}))

## 12 · Retrospective

The lifecycle is complete. Here is where the wall-clock time went on this machine, and what hurt.

In [ ]:
t = pd.Series(timings).rename('seconds').round(1).to_frame()
t['share'] = (t.seconds / t.seconds.sum()).map('{:.0%}'.format)
print(t.to_string()); print(f'\ntotal compute shown: {t.seconds.sum():.0f}s on {os.cpu_count()} CPU cores, GPU: {torch.cuda.is_available()}')

### What we learned about the *problem*
* Accuracy lied; place-based splits told the truth, and the same kind of model went from **0.89 → 0.80 → 0.48** as the test got more honest.
* The model works **inside the region it was trained on** and fails silently **outside** it. A gate and a serving-time warning both exist for that reason.
* A **quality-only gate would have shipped a bad map.** The prediction-stability check caught a model that had learned *one year's weather* instead of irrigation.
  Fixing it meant *removing* features, and it cost no accuracy.
* The 2024 map gives analysts *leads*, not conclusions; the labels are weak and no 2024 ground truth exists.
* The neural net did not beat a random forest on this small table. That is a fair result, and it is worth knowing before scaling anything up.

### What was painful about *doing* it this way
1. **Everything lived in one notebook's memory.** If the kernel dies we lose the state. Nothing is versioned (data, code, parameters, model).
2. **The 10 tuning trials ran one after another**, though they are fully independent. On a team, or with a bigger model, this is hours.
3. **One process, one machine.** Larger data or bigger networks would need many cores or GPUs, and sharing them fairly with colleagues is a scheduling problem we never solved.
4. **The "pipeline" is a Python function in a notebook.** Re-running it next year means someone remembering to open this file.
5. **Handing over the model = emailing a folder.** The serving code is a class in a notebook; a real app needs a stable network endpoint.
6. **Data came from a one-off download** with credentials tied to one person's cloud project.
7. **Nobody can tell which data, code and settings produced `cropwatch_v2`** except by reading this notebook top to bottom.

Hold on to that list: it is the *reason* the next layer of tooling exists. Each item is a gap this story has now made concrete.